# 03 · Continuation vs fade

Retention distributions, detection delay vs retention, the initial move vs
the 105-minute return, split by liquidity. Every field is read straight
from the persisted feature row.

> **This notebook is a research client, not a pipeline.** It contains no SQL, no
> threshold, and no classification rule. Every number comes from
> `afterhours_lab.research`, which reads persisted features computed once by
> `afterhours_lab.reactions`. Nothing here writes to the database — the pool is
> opened read-only.

In [ ]:
import datetime as dt

from afterhours_lab.research import (
    EventFilter,
    fetch_cohort,
    fetch_event_detail,
    fetch_class_distribution,
    fetch_monthly_counts,
    to_csv,
    to_jsonl,
    to_pandas,
    to_polars,
    write_parquet,
)
from afterhours_lab.research.notebook import (
    research_pool,
    describe_filter,
    describe_cohort,
    show_cohort,
    development_split,
)

# Jupyter already runs an event loop, so `await` works at cell top level.
pool = await research_pool()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
today = dt.date.today()
cohort_filter = EventFilter(
    date_from=today - dt.timedelta(days=365),
    date_to=today,
    analysis_statuses=('complete',),
    limit=5000,
)
async with pool.acquire() as conn:
    cohort = show_cohort(await fetch_cohort(conn, cohort_filter))
df = to_pandas(cohort.rows)
print(len(df), 'analyzed events')

## Retention distribution

`retention` is `return_105m / initial_return` as stored — > 1 means the
move extended, < 0 means it reversed.

In [ ]:
ret = df['retention'].dropna()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ret.clip(-2, 3), bins=40)
ax.set_xlabel('retention (clipped to [-2, 3] for display)')
ax.set_ylabel('events')
ax.set_title('retention of the initial reaction')
plt.tight_layout()

## Detection delay vs retention

In [ ]:
sub = df.dropna(subset=['detection_delay_minutes', 'retention'])
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(sub['detection_delay_minutes'], sub['retention'].clip(-2, 3), alpha=0.5)
ax.set_xlabel('detection delay (minutes)')
ax.set_ylabel('retention (clipped)')
ax.set_title('does a later signal retain less?')
plt.tight_layout()

## Initial move vs 105-minute return

In [ ]:
sub = df.dropna(subset=['initial_return', 'return_105m'])
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(sub['initial_return'], sub['return_105m'], alpha=0.5)
lim = max(sub['initial_return'].abs().max(), sub['return_105m'].abs().max()) * 1.05
ax.plot([-lim, lim], [-lim, lim], linestyle='--')
ax.set_xlabel('initial_return (%)')
ax.set_ylabel('return_105m (%)')
ax.set_title('on the 45° line = perfectly retained')
plt.tight_layout()

## Split by liquidity

Bucket on `volume_105m` (in-sample quantiles) and look at median retention
per bucket.

In [ ]:
import pandas as pd

liq = df.dropna(subset=['volume_105m', 'retention']).copy()
liq['liquidity_bucket'] = pd.qcut(
    liq['volume_105m'],
    q=4,
    labels=['low', 'mid-low', 'mid-high', 'high'],
    duplicates='drop',
)
liq.groupby('liquidity_bucket', observed=True)['retention'].agg(['count', 'median'])

In [ ]:
await pool.close()